In [1]:
from own_graph_library import Node
from own_graph_library import Edge
import numpy as np
import networkx as nx
import plotly.graph_objects as go

In [2]:
"""
Graph class implimented here in a odd way, possible to import as well but we shall see how it goes
"""

class Graph(object):
    def __init__(self):
        self.nodes = []
        self.edges = []
        self.adj_matrix = None # Adj. Matrix construction later, innit as None

    def add_node(self, index, label, movie_id):
        index = len(self.nodes)
        node = Node(index, label, movie_id)
        self.nodes.append(node)
        return node

    def add_edge(self, source, target, label, movie_id, weight):
        edge = Edge(source, target, label, movie_id, weight)
        self.edges.append(edge)
        self.nodes[source].links.append(edge)  # link from source to target


    def fill_adj_list(self):
        """
        ============== DEPREICATED ==============
        Takes the graph currently full of nodes and edges, and inserts the edges to the right node. Whole edge object is inserted for ease of use later.

        Only returns False when there is an error.

        """
        for edge in self.edges:
            try:
                self.nodes[edge.source].links.append(edge)
            except Exception as exception:
                print(f"Exception {exception} occured when insertinf edge")
                return exception

    def generate_adj_matrix(self):
        """
        Takes a graph of non-empty nodes, and non empty adj lists and fills in the adj_matrix, retruns a NxN 2d python list of floats
        """
        
        if self.nodes == []:
            self.adj_matrix = []
            return "No nodes, no adj matrix"
        

        try:
            n = len(self.nodes)
            self.adj_matrix = [[0 for i in range(n)] for j in range(n)]
            for i in range(n):
                for edge in self.nodes[i].links:
                    self.adj_matrix[i][edge.target] = edge.weight
        except Exception as exception:
            return exception
        
    def export_to_networkx(self):
        """
        Converts the current graph to a NetworkX graph.
        Returns a directed graph (DiGraph) object that can be used with NetworkX functionality.
        """
        # TODO!
        # Check for if graph is directed or undirected/ weighted unweighted
        
        G = nx.DiGraph()

        # Add nodes to the NetworkX graph with node attributes
        for node in self.nodes:
            G.add_node(node.index, label=node.label, movie_id=node.movie_id)

        # Add edges to the NetworkX graph with edge attributes
        for edge in self.edges:
            G.add_edge(
                edge.source, 
                edge.target, 
                weight=edge.weight, 
                label=edge.label,
                movie_id=edge.movie_id
            )
        
        return G
    # can refactor here a add node function that works with the 


In [3]:
main_graph = Graph()

In [4]:
# loading in node data:
with open("nodes.csv") as f:
    for line in f:
        if line[0] != "#":
            temp_holder = line.strip().split(",")
            new_node = main_graph.add_node(int(temp_holder[0]),temp_holder[1],int(temp_holder[2]))


In [5]:
# loading in edges data:
with open("edges.csv") as f:
    for line in f:
        if line[0] != "#":
            temp_holder = line.strip().split(",")
            new_edge = main_graph.add_edge(int(temp_holder[0]),int(temp_holder[1]),int(temp_holder[2]),int(temp_holder[3]),float(temp_holder[4]))


In [6]:
main_graph.generate_adj_matrix()

In [7]:
# Sample print of the first node with different out degree and out strength

print(main_graph.nodes[14])

index: 14, label: BOY #3, movie_id 316, out degree 3, out strength: 4.0


### Week 1 material:
---

(a,b,c) We picked the graph for Forrest Gump, and drew the graph here using networkX and ploty. We implimented our custom graph library to practise and write some of the math functions to algos from scratch

In [8]:
def draw_graph(main_graph):
    """
    Draws the graph using Plotly and NetworkX.
    I caved, I am too lazy to also write my own python GUI library and a custom spacer for nodes, so I stole networkX's code in this area
    """
    G = nx.DiGraph()  
    
    for node in main_graph.nodes:
        G.add_node(node.index, label=node.label)

    for edge in main_graph.edges:
        G.add_edge(edge.source, edge.target, weight=edge.weight, label=edge.label)

    pos = nx.spring_layout(G) 

    # Create Plotly graph
    edge_x = []
    edge_y = []
    edge_text = []
    for edge in main_graph.edges:
        x0, y0 = pos[edge.source]
        x1, y1 = pos[edge.target]
        edge_x.append(x0)
        edge_x.append(x1)
        edge_y.append(y0)
        edge_y.append(y1)
        edge_text.append(edge.label)

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        line=dict(width=0.5, color='gray'),
        hoverinfo='text',
        text=edge_text,
        mode='lines'
    )

    node_x = []
    node_y = []
    node_text = []
    for node in main_graph.nodes:
        x, y = pos[node.index]
        node_x.append(x)
        node_y.append(y)
        node_text.append(str(node))

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers",
        hoverinfo="text",
        text=node_text,
        marker=dict(
            showscale=True,
            colorscale="YlGnBu",
            size=10,
            colorbar=dict(thickness=15, title="Node connections", xanchor="left", titleside="right")
        )
    )

    # Create layout
    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(
                        showlegend=False,
                        hovermode='closest',
                        title='Network Graph Visualization',
                        xaxis=dict(showgrid=False, zeroline=False),
                        yaxis=dict(showgrid=False, zeroline=False),
                        plot_bgcolor='white'
                    ))
    fig.show()

In [9]:
draw_graph(main_graph)

(d) Compute the number of nodes,edges, average degree and the density. Comment.

In [ ]:
# Trivial computations
print(len(main_graph.nodes)) # no. of nodes
print(len(main_graph.edges)) # no. of edges


94
271


In [11]:
# Average degree out

# Helper function
def count_non_0(arr):
    count = 0
    for i in arr:
        if i > 0:
            count += 1
    return count

def get_avg_degree(main_graph_adj):
    """
    Takes in a 2D array, the adj. matrix of the graph, and returns     
    """
    avg_output = 0
    total_nodes = len(main_graph_adj)
    for i in range(total_nodes):
        non_zero = count_non_0(main_graph_adj[i])
        avg_output += non_zero/total_nodes
    return avg_output

# to deploy this function to calculate the in and out degree we can follow the proposition:
# Given A as Adj. matrix to directed graph G, A^T (A transpose) must be G' where all the directions of the arrows are reversed
# following this;
#  sum A_i = out degree 
#  sum A^T_i = in degree 

# in degree

avg_out_degree = get_avg_degree(main_graph.adj_matrix)

adj_matrix_transpose = np.array(main_graph.adj_matrix)
adj_matrix_transpose = np.transpose(adj_matrix_transpose)

avg_in_degree = get_avg_degree(adj_matrix_transpose)

print(f"average in degree: {avg_in_degree}")
print(f"average out degree: {avg_out_degree}")


average in degree: 2.882978723404256
average out degree: 2.882978723404253


In [17]:
# Density
def density_calc(main_graph):
    edges = len(main_graph.edges)
    nodes = len(main_graph.nodes)
    density = edges/(nodes*(nodes-1))
    return density

print(f"Density: {density_calc(main_graph)}")

Density: 0.030999771219400594


#### Comments:
 - Almost the same avg. in and out degree
 - quite low density, even though edges is almost 2x the density
 - avg in degree and out degree are both close to ~2.8, despite forrest gump being the main character (in which we expect a very hgih connections), indicating huge number of supplemantry cast/charaters 